In [2]:
import pandas as pd
import numpy as np

In [177]:
data_train = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/Train.csv")
data_train.head()

,ID,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,Depth_cm,Al,B,...,Fe,Mg,Mn,N,ph,P,K,Na,S,Zn
0,O2TONM,35.18756,-8.62390,01/01/2008,31/12/2018,50,20,20-50,1109.856,NaN,...,92.366,200.601,107.257,2.24,5.942,NaN,283.103,NaN,NaN,NaN
1,BQLUK6,35.18558,-8.62300,01/01/2008,31/12/2018,50,20,20-50,1168.364,NaN,...,115.923,197.771,90.005,1.57,5.722,NaN,215.459,NaN,NaN,NaN
2,LSET8M,35.18579,-8.62221,01/01/2008,31/12/2018,50,20,20-50,1137.113,NaN,...,78.709,188.114,120.433,1.02,5.510,NaN,398.656,NaN,NaN,NaN
3,LEEL7I,35.18266,-8.62177,01/01/2008,31/12/2018,50,20,20-50,1117.349,NaN,...,127.527,156.417,112.036,1.12,5.817,NaN,267.354,NaN,NaN,NaN
4,LDNGO2,35.12984,-8.62005,01/01/2008,31/12/2018,50,20,20-50,1219.203,NaN,...,77.542,114.809,57.906,1.19,4.980,NaN,229.682,NaN,NaN,NaN


In [4]:
features = data_train.iloc[:, :8]
features.head()

,ID,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,Depth_cm
0,O2TONM,35.18756,-8.62390,01/01/2008,31/12/2018,50,20,20-50
1,BQLUK6,35.18558,-8.62300,01/01/2008,31/12/2018,50,20,20-50
2,LSET8M,35.18579,-8.62221,01/01/2008,31/12/2018,50,20,20-50
3,LEEL7I,35.18266,-8.62177,01/01/2008,31/12/2018,50,20,20-50
4,LDNGO2,35.12984,-8.62005,01/01/2008,31/12/2018,50,20,20-50


In [176]:
data_test = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/TestSet.csv")
data_test.head()

,ID,Latitude,Longitude,Depth_cm,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,-0.746,37.094,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,DCC6DM,-0.785,37.178,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,T50LK1,-0.629,37.126,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,FNLYT0,-0.351,35.308,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,FP5E12,-1.894,36.987,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
# mergind train and test data
df = pd.concat([data_train, data_test], ignore_index=True)
df = df[["ID", "Longitude", "Latitude"]]
df.shape

(50368, 3)

# FE

## Country names

In [12]:
import reverse_geocoder as rg
import pycountry

In [13]:
coordinates = list(zip(df["Latitude"], df["Longitude"]))
locations = rg.search(coordinates)
locations_df = pd.DataFrame(locations)
locations_df.head()

Loading formatted geocoded file...


,lat,lon,name,admin1,admin2,cc
0,-8.73333,35.28333,Makungu,Iringa,,TZ
1,-8.73333,35.28333,Makungu,Iringa,,TZ
2,-8.73333,35.28333,Makungu,Iringa,,TZ
3,-8.73333,35.28333,Makungu,Iringa,,TZ
4,-8.73333,35.28333,Makungu,Iringa,,TZ


In [14]:
def get_country_name(country_code):
    try:
        country = pycountry.countries.get(alpha_2=country_code)
        return country.name
    except:
        return "Unknown"

In [15]:
df["Country"] = locations_df["cc"].apply(get_country_name)
df.head()

,ID,Longitude,Latitude,Country
0,O2TONM,35.18756,-8.62390,"Tanzania, United Republic of"
1,BQLUK6,35.18558,-8.62300,"Tanzania, United Republic of"
2,LSET8M,35.18579,-8.62221,"Tanzania, United Republic of"
3,LEEL7I,35.18266,-8.62177,"Tanzania, United Republic of"
4,LDNGO2,35.12984,-8.62005,"Tanzania, United Republic of"


In [16]:
df["Country"] = df["Country"].replace("Tanzania, United Republic of", "Tanzania")
df.head()

,ID,Longitude,Latitude,Country
0,O2TONM,35.18756,-8.62390,Tanzania
1,BQLUK6,35.18558,-8.62300,Tanzania
2,LSET8M,35.18579,-8.62221,Tanzania
3,LEEL7I,35.18266,-8.62177,Tanzania
4,LDNGO2,35.12984,-8.62005,Tanzania


## Nutrients by country

In [17]:
nutrients = pd.read_csv("../datasets/Environment_Cropland_nutrient_budget_E_Africa.csv")
nutrients.head()

,Area Code,Area Code (M49),Area,Item Code,Item,Element Code,Element,Unit,Y1961,Y1961F,...,Y2020N,Y2021,Y2021F,Y2021N,Y2022,Y2022F,Y2022N,Y2023,Y2023F,Y2023N
0,4,'012,Algeria,5087,Mineral fertilizers,7275,Cropland nitrogen,t,9000.0000,X,...,NaN,71800.0000,E,NaN,81800.0000,E,NaN,90100.0000,E,NaN
1,4,'012,Algeria,5087,Mineral fertilizers,7276,Cropland nitrogen per unit area,kg/ha,1.2737,E,...,NaN,8.4760,E,NaN,9.6016,E,NaN,10.5532,E,NaN
2,4,'012,Algeria,5087,Mineral fertilizers,7280,Cropland phosphorus,t,9790.3800,E,...,NaN,19639.6200,E,NaN,24171.8400,E,NaN,23622.4800,E,NaN
3,4,'012,Algeria,5087,Mineral fertilizers,7281,Cropland phosphorus per unit area,kg/ha,1.3856,E,...,NaN,2.3185,E,NaN,2.8373,E,NaN,2.7668,E,NaN
4,4,'012,Algeria,5087,Mineral fertilizers,7282,Cropland potassium,t,12242.5000,E,...,NaN,24734.0000,E,NaN,26352.5000,E,NaN,30295.0000,E,NaN


In [18]:
columns = ["Area", "Item", "Element", "Unit"]
years = ["Y" + str(year) for year in range(2008, 2019)]
columns.extend(years)
nutrients = nutrients[columns]
nutrients.head()

,Area,Item,Element,Unit,Y2008,Y2009,Y2010,Y2011,Y2012,Y2013,Y2014,Y2015,Y2016,Y2017,Y2018
0,Algeria,Mineral fertilizers,Cropland nitrogen,t,20051.0000,37100.0000,60100.0000,56300.0000,72600.0000,69800.0000,98150.0000,101550.0000,91750.0000,87900.0000,109600.0000
1,Algeria,Mineral fertilizers,Cropland nitrogen per unit area,kg/ha,2.3870,4.4172,7.1454,6.6857,8.6116,8.2748,11.6302,12.0001,10.8998,10.3614,12.8994
2,Algeria,Mineral fertilizers,Cropland phosphorus,t,6721.8120,19149.1200,12870.7200,14322.6000,22563.0000,28566.7200,31490.1000,26016.1200,23387.0400,22582.6200,25564.8600
3,Algeria,Mineral fertilizers,Cropland phosphorus per unit area,kg/ha,0.8002,2.2799,1.5302,1.7008,2.6764,3.3866,3.7314,3.0743,2.7784,2.6620,3.0089
4,Algeria,Mineral fertilizers,Cropland potassium,t,11980.6350,17513.0000,21248.0000,21165.0000,24900.0000,27846.5000,30585.5000,37059.5000,23198.5000,24277.5000,28718.0000


In [19]:
nutrients = nutrients[nutrients["Item"] == "Nutrient balance"]
nutrients.head()

,Area,Item,Element,Unit,Y2008,Y2009,Y2010,Y2011,Y2012,Y2013,Y2014,Y2015,Y2016,Y2017,Y2018
50,Algeria,Nutrient balance,Cropland nitrogen,t,39667.9061,-22450.4098,17644.5822,7555.0066,6919.5023,2153.3936,61887.3995,57804.6506,52046.9558,50167.8480,21249.8251
51,Algeria,Nutrient balance,Cropland nitrogen per unit area,kg/ha,4.7224,-2.6730,2.0978,0.8972,0.8208,0.2553,7.3333,6.8308,6.1831,5.9136,2.5010
52,Algeria,Nutrient balance,Cropland phosphorus,t,-2479.5653,-6439.8979,-8890.8782,-10702.7834,-4567.6425,-404.2307,8811.0021,792.8569,-1206.6284,-2074.9548,-10054.3113
53,Algeria,Nutrient balance,Cropland phosphorus per unit area,kg/ha,-0.2952,-0.7667,-1.0571,-1.2710,-0.5418,-0.0479,1.0441,0.0937,-0.1433,-0.2446,-1.1833
54,Algeria,Nutrient balance,Cropland potassium,t,-12635.5515,-33582.0078,-29597.7075,-35870.4380,-38529.6503,-42322.5519,-30118.4722,-26955.2135,-41074.6636,-40213.4938,-52090.7712


In [20]:
nutrients["Element"].unique()

<StringArray>
[                 'Cropland nitrogen',    'Cropland nitrogen per unit area',
                'Cropland phosphorus',  'Cropland phosphorus per unit area',
                 'Cropland potassium',   'Cropland potassium per unit area',
   'Cropland nitrogen use efficiency', 'Cropland phosphorus use efficiency',
  'Cropland potassium use efficiency']
Length: 9, dtype: str

In [21]:
nutrients = nutrients[nutrients["Element"].isin(["Cropland nitrogen per unit area", "Cropland phosphorus per unit area", "Cropland potassium per unit area"])]
nutrients["Y_AVG"] = nutrients[years].mean(axis=1)
nutrients.head()

,Area,Item,Element,Unit,Y2008,Y2009,Y2010,Y2011,Y2012,Y2013,Y2014,Y2015,Y2016,Y2017,Y2018,Y_AVG
51,Algeria,Nutrient balance,Cropland nitrogen per unit area,kg/ha,4.7224,-2.6730,2.0978,0.8972,0.8208,0.2553,7.3333,6.8308,6.1831,5.9136,2.5010,3.171118
53,Algeria,Nutrient balance,Cropland phosphorus per unit area,kg/ha,-0.2952,-0.7667,-1.0571,-1.2710,-0.5418,-0.0479,1.0441,0.0937,-0.1433,-0.2446,-1.1833,-0.401191
55,Algeria,Nutrient balance,Cropland potassium per unit area,kg/ha,-1.5042,-3.9983,-3.5189,-4.2596,-4.5703,-5.0174,-3.5689,-3.1853,-4.8796,-4.7402,-6.1309,-4.124873
110,Angola,Nutrient balance,Cropland nitrogen per unit area,kg/ha,7.3848,5.6716,7.7542,6.8697,11.7029,7.5706,13.0471,12.5781,9.7398,10.5133,8.5875,9.219964
112,Angola,Nutrient balance,Cropland phosphorus per unit area,kg/ha,-0.0211,-0.4694,-0.0401,0.4879,1.0390,-0.1206,0.1325,-0.0444,-0.3453,0.1213,-0.5499,0.017264


In [22]:
melt = nutrients.melt(
    id_vars=['Area', 'Element'],
    value_vars = years,
    var_name = 'Year',
    value_name = 'Value'
)

summary = melt.groupby(['Area', 'Element'])['Value'].agg(['min', 'max', 'mean', 'std']).reset_index()
summary.head()

,Area,Element,min,max,mean,std
0,Algeria,Cropland nitrogen per unit area,-2.6730,7.3333,3.171118,3.235247
1,Algeria,Cropland phosphorus per unit area,-1.2710,1.0441,-0.401191,0.671767
2,Algeria,Cropland potassium per unit area,-6.1309,-1.5042,-4.124873,1.201191
3,Angola,Cropland nitrogen per unit area,5.6716,13.0471,9.219964,2.465427
4,Angola,Cropland phosphorus per unit area,-0.5499,1.0390,0.017264,0.448005


In [23]:
nutrients_pivot = summary.pivot(index='Area', columns = 'Element', values = 'mean').reset_index()
nutrients_pivot.head()

Element,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area
0,Algeria,3.171118,-0.401191,-4.124873
1,Angola,9.219964,0.017264,-9.459173
2,Benin,-2.263545,-3.255536,-10.501491
3,Botswana,56.498218,2.144745,7.334409
4,Burkina Faso,3.875845,-1.323236,-0.840218


In [24]:
nutrients_pivot["Area"] = nutrients_pivot["Area"].replace("United Republic of Tanzania", "Tanzania")
nutrients_pivot["Area"].unique()

<StringArray>
[                         'Algeria',                           'Angola',
                            'Benin',                         'Botswana',
                     'Burkina Faso',                          'Burundi',
                       'Cabo Verde',                         'Cameroon',
         'Central African Republic',                             'Chad',
                          'Comoros',                            'Congo',
                    'Côte d'Ivoire', 'Democratic Republic of the Congo',
                         'Djibouti',                            'Egypt',
                'Equatorial Guinea',                          'Eritrea',
                         'Eswatini',                         'Ethiopia',
                     'Ethiopia PDR',                            'Gabon',
                           'Gambia',                            'Ghana',
                           'Guinea',                    'Guinea-Bissau',
                            'Kenya', 

In [26]:
features_nut = df.merge(nutrients_pivot, left_on="Country", right_on="Area", how="left")
features_nut.head()

,ID,Longitude,Latitude,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area
0,O2TONM,35.18756,-8.62390,Tanzania,Tanzania,6.777309,-2.492418,-7.380091
1,BQLUK6,35.18558,-8.62300,Tanzania,Tanzania,6.777309,-2.492418,-7.380091
2,LSET8M,35.18579,-8.62221,Tanzania,Tanzania,6.777309,-2.492418,-7.380091
3,LEEL7I,35.18266,-8.62177,Tanzania,Tanzania,6.777309,-2.492418,-7.380091
4,LDNGO2,35.12984,-8.62005,Tanzania,Tanzania,6.777309,-2.492418,-7.380091


In [27]:
features_nut[["Cropland nitrogen per unit area", "Cropland phosphorus per unit area", "Cropland potassium per unit area"]].isna().sum()

Cropland nitrogen per unit area      0
Cropland phosphorus per unit area    0
Cropland potassium per unit area     0
dtype: int64

## Worldclim features

In [11]:
df = features_nut.drop(columns=['Longitude', 'Latitude', 'start_date', 'end_date', 'horizon_lower', 'horizon_upper', 'Depth_cm'])
df.to_csv('../features/olek-features.csv', index=False)

NameError: name 'features_nut' is not defined

In [28]:
df = features_nut
df.head()

,ID,Longitude,Latitude,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area
0,O2TONM,35.18756,-8.62390,Tanzania,Tanzania,6.777309,-2.492418,-7.380091
1,BQLUK6,35.18558,-8.62300,Tanzania,Tanzania,6.777309,-2.492418,-7.380091
2,LSET8M,35.18579,-8.62221,Tanzania,Tanzania,6.777309,-2.492418,-7.380091
3,LEEL7I,35.18266,-8.62177,Tanzania,Tanzania,6.777309,-2.492418,-7.380091
4,LDNGO2,35.12984,-8.62005,Tanzania,Tanzania,6.777309,-2.492418,-7.380091


In [29]:
import rasterio
import os
import glob

In [30]:
def create_avg_raster(input, output):
    tif_files = glob.glob(os.path.join(input, "*.tif"))

    if not tif_files:
        print("No .tif files found in selected directory.")
        return
    
    with rasterio.open(tif_files[0]) as src:
        meta = src.meta.copy()
        base_shape = src.shape
        print(f"Base shape: {base_shape}")
    
    data_to_average = []
    for tif in tif_files:
        with rasterio.open(tif) as src:
            if src.shape != base_shape:
                print(f"Skipping {tif} due to shape mismatch.")
                continue
            tif_data = src.read(1).astype(np.float32)
            data_to_average.append(tif_data)

    if not data_to_average:
        print("No valid .tif files to average.")
        return
    
    data_stack = np.stack(data_to_average)
    avg_data = np.mean(data_stack, axis=0)

    meta.update({
        "dtype": 'float32',
        "count": 1
    })

    with rasterio.open(output, 'w', **meta) as dst:
        dst.write(avg_data, 1)
        
    print(f"Success. Saved as: {output}")

In [ ]:
tmin_data_dir = "../../worldclim/tmin_2008-2018"
output_tif = "../datasets/tmin_avg_2008-2018.tif"
create_avg_raster(tmin_data_dir, output_tif)

Base shape: (2160, 4320)
Success. Saved as: ../datasets/tmin_avg_2008-2018.tif


In [ ]:
tmax_data_dir = "../../worldclim/tmax_2008-2018"
output_tif = "../datasets/tmax_avg_2008-2018.tif"
create_avg_raster(tmax_data_dir, output_tif)

Base shape: (2160, 4320)
Success. Saved as: ../datasets/tmax_avg_2008-2018.tif


In [ ]:
prec_data_dir = "../../worldclim/prec_2008-2018"
output_tif = "../datasets/prec_avg_2008-2018.tif"
create_avg_raster(prec_data_dir, output_tif)

Base shape: (2160, 4320)
Success. Saved as: ../datasets/prec_avg_2008-2018.tif


In [31]:
def get_pixel_value(coords, files):
    all_res = []
    for idx, (lon, lat) in enumerate(coords):
        data_row = {'Point_id': idx, 'Longitude': lon, 'Latitude': lat}
        
        
        for feature_name, file_path in files.items():
            try:
                with rasterio.open(file_path) as src:
                    if not (src.bounds.left <= lon <= src.bounds.right and src.bounds.bottom <= lat <= src.bounds.top):
                        continue 
                    
                    for layers_vals in src.sample([(lon, lat)]):
                        for n_layer, value in enumerate(layers_vals, start=1):
                            col_name = f"Band{n_layer}_{feature_name}"
                            data_row[col_name] = value
            
            except Exception as e:
                print(f"Błąd przy odczycie pliku {file_path}: {e}")
                
        all_res.append(data_row)
    
    df = pd.DataFrame(all_res)
    return df


In [39]:
points = df[["Longitude", "Latitude"]].values
files = {
    "tmin_avg": "../datasets/tmin_avg_2008-2018.tif",
    "tmax_avg": "../datasets/tmax_avg_2008-2018.tif",
    "prec_avg": "../datasets/prec_avg_2008-2018.tif"
}

In [40]:
worldclim_df = get_pixel_value(points, files)
worldclim_df = worldclim_df.drop(columns=['Point_id', 'Longitude', 'Latitude'])
worldclim_df.head()

,Band1_tmin_avg,Band1_tmax_avg,Band1_prec_avg
0,12.727273,22.628788,123.268784
1,12.727273,22.628788,123.268784
2,12.727273,22.628788,123.268784
3,12.727273,22.628788,123.268784
4,12.674242,23.077652,110.355507


In [41]:
worldclim_df.columns = ["tmin_avg", "tmax_avg", "prec_avg"]
worldclim_df.head()

,tmin_avg,tmax_avg,prec_avg
0,12.727273,22.628788,123.268784
1,12.727273,22.628788,123.268784
2,12.727273,22.628788,123.268784
3,12.727273,22.628788,123.268784
4,12.674242,23.077652,110.355507


In [42]:
tmin = worldclim_df["tmin_avg"]
tmin.max() - tmin.min()

np.float32(20.431818)

In [44]:
worldclim_df.shape

(50368, 3)

In [36]:
mateusz_features = pd.read_csv("../features/mateusz-features.csv")
mateusz_features.shape

(14, 27)

In [37]:
olek_features = pd.read_csv("../features/olek-features.csv")
olek_features.shape

(44298, 9)

In [46]:
olek_features.head()

,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.35551


In [19]:
features_comb = pd.read_csv('../features/olek-features.csv')
features_comb.head()

,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091


In [45]:
features_final = df.join(worldclim_df)
features_final.head()

,ID,Longitude,Latitude,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg
0,O2TONM,35.18756,-8.62390,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.268784
1,BQLUK6,35.18558,-8.62300,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.268784
2,LSET8M,35.18579,-8.62221,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.268784
3,LEEL7I,35.18266,-8.62177,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.268784
4,LDNGO2,35.12984,-8.62005,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.355507


In [47]:
features_final.drop(columns=['Longitude', 'Latitude'], inplace=True)
features_final.head()

,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.268784
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.268784
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.268784
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.268784
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.355507


In [48]:
features_final.to_csv('../features/olek-features.csv', index=False)

## Earth Engine

In [1]:
import ee

In [3]:
ee.Initialize(project='lucknera')

In [19]:
lon = df["Longitude"]
lat = df["Latitude"]
points = list(zip(lon, lat))
points[:5]

[(35.18756, -8.6239),
 (35.18558, -8.623),
 (35.18579, -8.62221),
 (35.18266, -8.62177),
 (35.12984, -8.62005)]

In [20]:
df.head()

,ID,Longitude,Latitude
0,O2TONM,35.18756,-8.62390
1,BQLUK6,35.18558,-8.62300
2,LSET8M,35.18579,-8.62221
3,LEEL7I,35.18266,-8.62177
4,LDNGO2,35.12984,-8.62005


In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50368 entries, 0 to 50367
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID         50368 non-null  str    
 1   Longitude  50368 non-null  float64
 2   Latitude   50368 non-null  float64
dtypes: float64(2), str(1)
memory usage: 1.2 MB


In [21]:
df.to_csv('../features/olek-sentinel-features.csv', index=False)

In [ ]:
# projects/lucknera/assets/olek-sentinel-features

In [43]:
df1, df2, df3 = df.iloc[:25000], df.iloc[25000:40000], df.iloc[40000:]

In [39]:
df1.to_csv('../features/olek-first-sentinel-features.csv', index=False)

In [45]:
df2.to_csv('../features/olek-second-sentinel-features.csv', index=False)

In [46]:
df3.to_csv('../features/olek-third-sentinel-features.csv', index=False)

In [ ]:
points = ee.FeatureCollection("projects/lucknera/assets/olek-third-sentinel-features")

clear_points = points.filter(ee.Filter.notNull(['Longitude', 'Latitude']))

def create_geometry(feature):
    lon = ee.Number(feature.get('Longitude'))
    lat = ee.Number(feature.get('Latitude'))
    return feature.setGeometry(ee.Geometry.Point([lon, lat]))

punkty_geo = clear_points.map(create_geometry)



colection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate('2019-01-01', '2020-12-31') \
    .filterBounds(punkty_geo) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])

median_img = colection.median()


extraction = median_img.reduceRegions(
    collection=punkty_geo,
    reducer=ee.Reducer.first(),
    scale=10,
    tileScale=16 
)


export = ee.batch.Export.table.toDrive(
    collection=extraction,
    description='Wyniki_Gleba_Third_Part', 
    folder='EarthEngine_Dane',              
    fileNamePrefix='dane_satelitarne_gleba_50k_third_part', 
    fileFormat='CSV'
)

export.start()
print("Sending safe export task to GEE...")

Wysyłam bezpieczne zadanie do GEE...


In [50]:
EE_df1 = pd.read_csv('../features/EE_data/dane_satelitarne_gleba_50k.csv')
print(EE_df1.shape)
EE_df1.head()

(25000, 11)


,system:index,B11,B12,B2,B3,B4,B8,ID,Latitude,Longitude,.geo
0,00000000000000001b6a,2275.5,1258.5,393.0,695.0,577.0,2782.0,JUZSWK,10.53952,-13.43310,"{""type"":""Point"",""coordinates"":[-13.43309974670..."
1,00000000000000004478,2275.5,1258.5,393.0,695.0,577.0,2782.0,B5RDUR,10.53952,-13.43310,"{""type"":""Point"",""coordinates"":[-13.43309974670..."
2,0000000000000000444f,2327.5,1422.0,381.0,605.5,632.0,2300.0,IMQ2UL,10.51887,-13.43300,"{""type"":""Point"",""coordinates"":[-13.43299961090..."
3,00000000000000004450,2327.5,1422.0,381.0,605.5,632.0,2300.0,GAP8LB,10.51887,-13.43300,"{""type"":""Point"",""coordinates"":[-13.43299961090..."
4,00000000000000001b6d,2530.5,1502.5,418.5,720.5,807.5,2624.0,3EYUL8,10.54100,-13.43276,"{""type"":""Point"",""coordinates"":[-13.43276023864..."


In [51]:
EE_df2 = pd.read_csv('../features/EE_data/dane_satelitarne_gleba_50k_second_part.csv')
print(EE_df2.shape)
EE_df2.head()

(15000, 11)


,system:index,B11,B12,B2,B3,B4,B8,ID,Latitude,Longitude,.geo
0,00000000000000001fe0,3889.111111,3374.000000,1040.800000,1383.333333,1851.500000,2796.750000,SA5P3U,-16.19618,14.87336,"{""type"":""Point"",""coordinates"":[14.873359680175..."
1,00000000000000001fcc,2906.333333,2059.857143,688.272727,907.920000,1102.000000,2286.909091,L0G9X7,-16.19925,14.87618,"{""type"":""Point"",""coordinates"":[14.876179695129..."
2,00000000000000001fc7,3228.033333,2435.166667,719.600000,911.294118,1149.837607,2253.500000,CLLXS9,-16.20040,14.87656,"{""type"":""Point"",""coordinates"":[14.876560211181..."
3,00000000000000001fc5,3056.538462,2465.784615,626.666667,847.500000,977.833333,2221.692308,DRJ0QX,-16.20118,14.87673,"{""type"":""Point"",""coordinates"":[14.876729965209..."
4,00000000000000001fdb,4309.000000,3787.166667,983.500000,1333.272727,1841.166667,2904.800000,D6AZIW,-16.19691,14.87715,"{""type"":""Point"",""coordinates"":[14.877149581909..."


In [52]:
EE_df3 = pd.read_csv('../features/EE_data/dane_satelitarne_gleba_50k_third_part.csv')
print(EE_df3.shape)
EE_df3.head()

(10368, 11)


,system:index,B11,B12,B2,B3,B4,B8,ID,Latitude,Longitude,.geo
0,00000000000000001852,2327.5,1422.0,381.0,605.5,632.0,2300.0,Q7QJ6M,10.51887,-13.43300,"{""type"":""Point"",""coordinates"":[-13.43299961090..."
1,00000000000000001d07,2280.5,1304.5,395.0,661.5,630.5,2717.5,5XD1YT,10.53776,-13.43185,"{""type"":""Point"",""coordinates"":[-13.43185043334..."
2,000000000000000027b3,2573.0,1680.0,422.0,683.0,752.5,2474.0,F7LO6G,10.51204,-13.42769,"{""type"":""Point"",""coordinates"":[-13.42768955230..."
3,00000000000000001850,2102.5,1128.5,339.0,597.0,425.0,2890.0,TQC7K7,10.51609,-13.42728,"{""type"":""Point"",""coordinates"":[-13.42728042602..."
4,00000000000000001d02,2098.0,1052.0,380.5,681.0,548.0,3108.0,ANAJ12,10.49820,-13.42662,"{""type"":""Point"",""coordinates"":[-13.42661952972..."


In [53]:
features_EE = pd.concat([EE_df1, EE_df2, EE_df3], ignore_index=True)
print(features_EE.shape)
features_EE.head()

(50368, 11)


,system:index,B11,B12,B2,B3,B4,B8,ID,Latitude,Longitude,.geo
0,00000000000000001b6a,2275.5,1258.5,393.0,695.0,577.0,2782.0,JUZSWK,10.53952,-13.43310,"{""type"":""Point"",""coordinates"":[-13.43309974670..."
1,00000000000000004478,2275.5,1258.5,393.0,695.0,577.0,2782.0,B5RDUR,10.53952,-13.43310,"{""type"":""Point"",""coordinates"":[-13.43309974670..."
2,0000000000000000444f,2327.5,1422.0,381.0,605.5,632.0,2300.0,IMQ2UL,10.51887,-13.43300,"{""type"":""Point"",""coordinates"":[-13.43299961090..."
3,00000000000000004450,2327.5,1422.0,381.0,605.5,632.0,2300.0,GAP8LB,10.51887,-13.43300,"{""type"":""Point"",""coordinates"":[-13.43299961090..."
4,00000000000000001b6d,2530.5,1502.5,418.5,720.5,807.5,2624.0,3EYUL8,10.54100,-13.43276,"{""type"":""Point"",""coordinates"":[-13.43276023864..."


In [54]:
features_EE.drop(columns=['system:index', 'Longitude', 'Latitude', '.geo'], inplace=True)

In [56]:
features_EE.head()

,B11,B12,B2,B3,B4,B8,ID
0,2275.5,1258.5,393.0,695.0,577.0,2782.0,JUZSWK
1,2275.5,1258.5,393.0,695.0,577.0,2782.0,B5RDUR
2,2327.5,1422.0,381.0,605.5,632.0,2300.0,IMQ2UL
3,2327.5,1422.0,381.0,605.5,632.0,2300.0,GAP8LB
4,2530.5,1502.5,418.5,720.5,807.5,2624.0,3EYUL8


In [57]:
olek_features = pd.read_csv('../features/olek-features.csv')
print(olek_features.shape)
olek_features.head()

(50368, 9)


,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.35551


In [60]:
df_merged = olek_features.merge(features_EE, on='ID', how='left')
print(df_merged.shape)
df_merged.head()

(50368, 15)


,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,B12,B2,B3,B4,B8
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,1287.0,699.0,243.0,364.0,276.0,1782.0
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3215.0,2426.0,789.0,1066.0,1140.0,2714.0
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3127.0,2220.0,697.0,943.0,1114.0,2484.0
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,2085.0,1370.0,499.0,691.0,750.0,2032.0
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.35551,2059.0,1076.0,324.0,436.0,377.0,2620.0


In [61]:
df_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 50368 entries, 0 to 50367
Data columns (total 15 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   ID                                 50368 non-null  str    
 1   Country                            50368 non-null  str    
 2   Area                               50368 non-null  str    
 3   Cropland nitrogen per unit area    50368 non-null  float64
 4   Cropland phosphorus per unit area  50368 non-null  float64
 5   Cropland potassium per unit area   50368 non-null  float64
 6   tmin_avg                           50368 non-null  float64
 7   tmax_avg                           50368 non-null  float64
 8   prec_avg                           50368 non-null  float64
 9   B11                                50368 non-null  float64
 10  B12                                50368 non-null  float64
 11  B2                                 50368 non-null  float64
 12  B

In [62]:
df_merged.to_csv('../features/olek-features.csv', index=False)

## SRTM features

In [14]:
import ee

In [15]:
ee.Initialize(project='lucknera')

In [16]:
srtm = ee.Image('USGS/SRTMGL1_003')

terrain = ee.Terrain.products(srtm)

In [ ]:
df.head()

,ID,Longitude,Latitude
0,O2TONM,35.18756,-8.62390
1,BQLUK6,35.18558,-8.62300
2,LSET8M,35.18579,-8.62221
3,LEEL7I,35.18266,-8.62177
4,LDNGO2,35.12984,-8.62005


In [34]:
def create_geometry(feature):
    lon = ee.Number(feature.get('Longitude'))
    lat = ee.Number(feature.get('Latitude'))
    return feature.setGeometry(ee.Geometry.Point([lon, lat]))

In [35]:
points = ee.FeatureCollection("projects/lucknera/assets/olek-sentinel-features")
points_geo = points.map(create_geometry)

In [38]:
extraction = terrain.sampleRegions(
    collection=points_geo,
    scale=30,
    geometries=True
)

In [40]:
task = ee.batch.Export.table.toDrive(
    collection=extraction,           
    description='Mines_Features_AF', 
    folder='EarthEngine_Dane',            
    fileNamePrefix='SRTM_data_features',   
    fileFormat='CSV')

In [41]:
task.start()

In [43]:
SRTM_data = pd.read_csv('../datasets/SRTM_data_features.csv')
print(SRTM_data.shape)
SRTM_data.head()

(50368, 9)


,system:index,ID,Latitude,Longitude,aspect,elevation,hillshade,slope,.geo
0,00000000000000001b6a_0,JUZSWK,10.53952,-13.4331,184,392,183,15,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,00000000000000004478_0,B5RDUR,10.53952,-13.4331,184,392,183,15,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,0000000000000000444f_0,IMQ2UL,10.51887,-13.4330,157,266,166,12,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,00000000000000004450_0,GAP8LB,10.51887,-13.4330,157,266,166,12,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,0000000000000000b492_0,Q7QJ6M,10.51887,-13.4330,157,266,166,12,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


In [44]:
SRTM_data.drop(columns=['system:index', 'Longitude', 'Latitude', '.geo'], inplace=True)
SRTM_data.head()

,ID,aspect,elevation,hillshade,slope
0,JUZSWK,184,392,183,15
1,B5RDUR,184,392,183,15
2,IMQ2UL,157,266,166,12
3,GAP8LB,157,266,166,12
4,Q7QJ6M,157,266,166,12


In [45]:
olek_features = pd.read_csv('../features/olek-features.csv')
olek_features.head()

,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,B12,B2,B3,B4,B8
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,1287.0,699.0,243.0,364.0,276.0,1782.0
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3215.0,2426.0,789.0,1066.0,1140.0,2714.0
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3127.0,2220.0,697.0,943.0,1114.0,2484.0
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,2085.0,1370.0,499.0,691.0,750.0,2032.0
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.35551,2059.0,1076.0,324.0,436.0,377.0,2620.0


In [46]:
features_comb = pd.merge(olek_features, SRTM_data, on='ID', how='left')
print(features_comb.shape)
features_comb.head()

(50368, 19)


,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,B12,B2,B3,B4,B8,aspect,elevation,hillshade,slope
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,1287.0,699.0,243.0,364.0,276.0,1782.0,192,1895,189,13
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3215.0,2426.0,789.0,1066.0,1140.0,2714.0,265,1884,215,11
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3127.0,2220.0,697.0,943.0,1114.0,2484.0,315,1888,195,7
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,2085.0,1370.0,499.0,691.0,750.0,2032.0,45,1898,163,8
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.35551,2059.0,1076.0,324.0,436.0,377.0,2620.0,278,1894,201,7


In [47]:
features_comb.to_csv('../features/olek-features.csv', index=False)

## Soil Grids features

In [48]:
import ee

In [49]:
ee.Initialize(project='lucknera')

In [50]:
sand = ee.Image('projects/soilgrids-isric/sand_mean')
clay = ee.Image('projects/soilgrids-isric/clay_mean')
soc = ee.Image('projects/soilgrids-isric/soc_mean')
ph = ee.Image('projects/soilgrids-isric/phh2o_mean')
cec = ee.Image('projects/soilgrids-isric/cec_mean')

In [51]:
soil_all = ee.Image.cat([sand, clay, soc, ph, cec])

In [52]:
def create_geometry(feature):
    lon = ee.Number(feature.get('Longitude'))
    lat = ee.Number(feature.get('Latitude'))
    return feature.setGeometry(ee.Geometry.Point([lon, lat]))

In [53]:
points = ee.FeatureCollection("projects/lucknera/assets/olek-sentinel-features")
points_geo = points.map(create_geometry)

In [55]:
extraction_soil = soil_all.sampleRegions(
    collection=points_geo,
    scale=250,
    geometries=True
)

In [56]:
task = ee.batch.Export.table.toDrive(
    collection=extraction_soil,
    description = 'Soil_Grid_Features',
    folder='EarthEngine_Dane',
    fileNamePrefix='SG_data_features',   
    fileFormat='CSV')

In [58]:
task.start()

In [59]:
SG_features = pd.read_csv('../datasets/SG_data_features.csv')
print(SG_features.shape)
SG_features.head()

(50214, 35)


,system:index,ID,Latitude,Longitude,cec_0-5cm_mean,cec_100-200cm_mean,cec_15-30cm_mean,cec_30-60cm_mean,cec_5-15cm_mean,cec_60-100cm_mean,...,sand_30-60cm_mean,sand_5-15cm_mean,sand_60-100cm_mean,soc_0-5cm_mean,soc_100-200cm_mean,soc_15-30cm_mean,soc_30-60cm_mean,soc_5-15cm_mean,soc_60-100cm_mean,.geo
0,00000000000000001b6a_0,JUZSWK,10.53952,-13.4331,277,286,271,276,273,281,...,448,610,454,410,111,214,116,456,97,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,00000000000000004478_0,B5RDUR,10.53952,-13.4331,277,286,271,276,273,281,...,448,610,454,410,111,214,116,456,97,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,0000000000000000444f_0,IMQ2UL,10.51887,-13.4330,253,258,251,252,250,255,...,452,645,447,471,211,289,236,502,212,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,00000000000000004450_0,GAP8LB,10.51887,-13.4330,253,258,251,252,250,255,...,452,645,447,471,211,289,236,502,212,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,0000000000000000b492_0,Q7QJ6M,10.51887,-13.4330,253,258,251,252,250,255,...,452,645,447,471,211,289,236,502,212,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


In [60]:
SG_features.drop(columns=['system:index', 'Longitude', 'Latitude', '.geo'], inplace=True)
SG_features.head()

,ID,cec_0-5cm_mean,cec_100-200cm_mean,cec_15-30cm_mean,cec_30-60cm_mean,cec_5-15cm_mean,cec_60-100cm_mean,clay_0-5cm_mean,clay_100-200cm_mean,clay_15-30cm_mean,...,sand_15-30cm_mean,sand_30-60cm_mean,sand_5-15cm_mean,sand_60-100cm_mean,soc_0-5cm_mean,soc_100-200cm_mean,soc_15-30cm_mean,soc_30-60cm_mean,soc_5-15cm_mean,soc_60-100cm_mean
0,JUZSWK,277,286,271,276,273,281,141,359,236,...,538,448,610,454,410,111,214,116,456,97
1,B5RDUR,277,286,271,276,273,281,141,359,236,...,538,448,610,454,410,111,214,116,456,97
2,IMQ2UL,253,258,251,252,250,255,126,397,242,...,552,452,645,447,471,211,289,236,502,212
3,GAP8LB,253,258,251,252,250,255,126,397,242,...,552,452,645,447,471,211,289,236,502,212
4,Q7QJ6M,253,258,251,252,250,255,126,397,242,...,552,452,645,447,471,211,289,236,502,212


In [66]:
SG_features.columns

Index(['ID', 'cec_0-5cm_mean', 'cec_100-200cm_mean', 'cec_15-30cm_mean',
       'cec_30-60cm_mean', 'cec_5-15cm_mean', 'cec_60-100cm_mean',
       'clay_0-5cm_mean', 'clay_100-200cm_mean', 'clay_15-30cm_mean',
       'clay_30-60cm_mean', 'clay_5-15cm_mean', 'clay_60-100cm_mean',
       'phh2o_0-5cm_mean', 'phh2o_100-200cm_mean', 'phh2o_15-30cm_mean',
       'phh2o_30-60cm_mean', 'phh2o_5-15cm_mean', 'phh2o_60-100cm_mean',
       'sand_0-5cm_mean', 'sand_100-200cm_mean', 'sand_15-30cm_mean',
       'sand_30-60cm_mean', 'sand_5-15cm_mean', 'sand_60-100cm_mean',
       'soc_0-5cm_mean', 'soc_100-200cm_mean', 'soc_15-30cm_mean',
       'soc_30-60cm_mean', 'soc_5-15cm_mean', 'soc_60-100cm_mean'],
      dtype='str')

In [68]:
deep_features = [col for col in SG_features.columns if '100' in col]
deep_features

['cec_100-200cm_mean',
 'cec_60-100cm_mean',
 'clay_100-200cm_mean',
 'clay_60-100cm_mean',
 'phh2o_100-200cm_mean',
 'phh2o_60-100cm_mean',
 'sand_100-200cm_mean',
 'sand_60-100cm_mean',
 'soc_100-200cm_mean',
 'soc_60-100cm_mean']

In [69]:
SG_features.drop(columns=deep_features, inplace=True)
print(SG_features.shape)
SG_features.head()

(50214, 21)


,ID,cec_0-5cm_mean,cec_15-30cm_mean,cec_30-60cm_mean,cec_5-15cm_mean,clay_0-5cm_mean,clay_15-30cm_mean,clay_30-60cm_mean,clay_5-15cm_mean,phh2o_0-5cm_mean,...,phh2o_30-60cm_mean,phh2o_5-15cm_mean,sand_0-5cm_mean,sand_15-30cm_mean,sand_30-60cm_mean,sand_5-15cm_mean,soc_0-5cm_mean,soc_15-30cm_mean,soc_30-60cm_mean,soc_5-15cm_mean
0,JUZSWK,277,271,276,273,141,236,362,138,51,...,51,51,613,538,448,610,410,214,116,456
1,B5RDUR,277,271,276,273,141,236,362,138,51,...,51,51,613,538,448,610,410,214,116,456
2,IMQ2UL,253,251,252,250,126,242,380,125,52,...,52,52,650,552,452,645,471,289,236,502
3,GAP8LB,253,251,252,250,126,242,380,125,52,...,52,52,650,552,452,645,471,289,236,502
4,Q7QJ6M,253,251,252,250,126,242,380,125,52,...,52,52,650,552,452,645,471,289,236,502


In [70]:
SG_features.columns

Index(['ID', 'cec_0-5cm_mean', 'cec_15-30cm_mean', 'cec_30-60cm_mean',
       'cec_5-15cm_mean', 'clay_0-5cm_mean', 'clay_15-30cm_mean',
       'clay_30-60cm_mean', 'clay_5-15cm_mean', 'phh2o_0-5cm_mean',
       'phh2o_15-30cm_mean', 'phh2o_30-60cm_mean', 'phh2o_5-15cm_mean',
       'sand_0-5cm_mean', 'sand_15-30cm_mean', 'sand_30-60cm_mean',
       'sand_5-15cm_mean', 'soc_0-5cm_mean', 'soc_15-30cm_mean',
       'soc_30-60cm_mean', 'soc_5-15cm_mean'],
      dtype='str')

In [71]:
olek_features = pd.read_csv('../features/olek-features.csv')
print(olek_features.shape)
olek_features.head()

(50368, 19)


,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,B12,B2,B3,B4,B8,aspect,elevation,hillshade,slope
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,1287.0,699.0,243.0,364.0,276.0,1782.0,192,1895,189,13
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3215.0,2426.0,789.0,1066.0,1140.0,2714.0,265,1884,215,11
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3127.0,2220.0,697.0,943.0,1114.0,2484.0,315,1888,195,7
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,2085.0,1370.0,499.0,691.0,750.0,2032.0,45,1898,163,8
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.35551,2059.0,1076.0,324.0,436.0,377.0,2620.0,278,1894,201,7


In [72]:
features_comb = pd.merge(olek_features, SG_features, on='ID', how='left')
print(features_comb.shape)
features_comb.head()

(50368, 39)


,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,...,phh2o_30-60cm_mean,phh2o_5-15cm_mean,sand_0-5cm_mean,sand_15-30cm_mean,sand_30-60cm_mean,sand_5-15cm_mean,soc_0-5cm_mean,soc_15-30cm_mean,soc_30-60cm_mean,soc_5-15cm_mean
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,1287.0,...,52.0,52.0,508.0,463.0,441.0,502.0,286.0,215.0,181.0,226.0
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3215.0,...,52.0,52.0,527.0,477.0,461.0,514.0,260.0,164.0,137.0,229.0
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3127.0,...,52.0,52.0,527.0,477.0,461.0,514.0,260.0,164.0,137.0,229.0
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,2085.0,...,52.0,52.0,514.0,490.0,448.0,508.0,270.0,158.0,124.0,225.0
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.35551,2059.0,...,52.0,52.0,544.0,501.0,491.0,528.0,233.0,125.0,112.0,190.0


In [75]:
features_comb.isnull().sum()

ID                                     0
Country                                0
Area                                   0
Cropland nitrogen per unit area        0
Cropland phosphorus per unit area      0
Cropland potassium per unit area       0
tmin_avg                               0
tmax_avg                               0
prec_avg                               0
B11                                    0
B12                                    0
B2                                     0
B3                                     0
B4                                     0
B8                                     0
aspect                                 0
elevation                              0
hillshade                              0
slope                                  0
cec_0-5cm_mean                       154
cec_15-30cm_mean                     154
cec_30-60cm_mean                     154
cec_5-15cm_mean                      154
clay_0-5cm_mean                      154
clay_15-30cm_mea

In [81]:
features_comb.fillna(SG_features.mean(numeric_only=True), inplace=True)

,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,...,phh2o_30-60cm_mean,phh2o_5-15cm_mean,sand_0-5cm_mean,sand_15-30cm_mean,sand_30-60cm_mean,sand_5-15cm_mean,soc_0-5cm_mean,soc_15-30cm_mean,soc_30-60cm_mean,soc_5-15cm_mean
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,1287.0,...,52.0,52.0,508.0,463.0,441.0,502.0,286.0,215.0,181.0,226.0
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3215.0,...,52.0,52.0,527.0,477.0,461.0,514.0,260.0,164.0,137.0,229.0
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3127.0,...,52.0,52.0,527.0,477.0,461.0,514.0,260.0,164.0,137.0,229.0
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,2085.0,...,52.0,52.0,514.0,490.0,448.0,508.0,270.0,158.0,124.0,225.0
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.35551,2059.0,...,52.0,52.0,544.0,501.0,491.0,528.0,233.0,125.0,112.0,190.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50363,YSP8V5,Nigeria,Nigeria,1.934491,-3.448673,-13.578155,21.151516,34.126892,89.80792,3828.0,...,62.0,62.0,614.0,543.0,520.0,604.0,122.0,70.0,50.0,81.0
50364,T2I1P5,Nigeria,Nigeria,1.934491,-3.448673,-13.578155,22.554924,33.077652,91.94679,3537.0,...,58.0,59.0,609.0,601.0,565.0,619.0,167.0,59.0,46.0,84.0
50365,HBG2O0,Nigeria,Nigeria,1.934491,-3.448673,-13.578155,22.761364,32.998108,108.79585,4000.0,...,54.0,55.0,658.0,656.0,619.0,682.0,132.0,63.0,38.0,72.0
50366,1UWDEP,Nigeria,Nigeria,1.934491,-3.448673,-13.578155,22.037878,33.676136,94.17880,3392.0,...,58.0,59.0,613.0,575.0,519.0,598.0,197.0,121.0,93.0,165.0


In [82]:
features_comb.info()

<class 'pandas.DataFrame'>
RangeIndex: 50368 entries, 0 to 50367
Data columns (total 39 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   ID                                 50368 non-null  str    
 1   Country                            50368 non-null  str    
 2   Area                               50368 non-null  str    
 3   Cropland nitrogen per unit area    50368 non-null  float64
 4   Cropland phosphorus per unit area  50368 non-null  float64
 5   Cropland potassium per unit area   50368 non-null  float64
 6   tmin_avg                           50368 non-null  float64
 7   tmax_avg                           50368 non-null  float64
 8   prec_avg                           50368 non-null  float64
 9   B11                                50368 non-null  float64
 10  B12                                50368 non-null  float64
 11  B2                                 50368 non-null  float64
 12  B

In [83]:
features_comb.to_csv('../features/olek-features.csv', index=False)

## ASTER features

In [84]:
points = ee.FeatureCollection("projects/lucknera/assets/olek-sentinel-features")
points_geo = points.map(create_geometry)

In [108]:
colection = ee.ImageCollection('ASTER/AST_L1T_003') \
    .filterDate('2000-01-01', '2007-12-31') \
    .filterBounds(points_geo) \
    .filter(ee.Filter.lt('CLOUDCOVER', 20))


In [109]:
median_img = colection.median()

In [110]:
aster_minerals = median_img.select(['B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B10', 'B11', 'B12', 'B13', 'B14'])

In [111]:
extraction = aster_minerals.sampleRegions(
    collection=points_geo,
    scale=30,
    geometries=True
)

In [112]:
task = ee.batch.Export.table.toDrive(
    collection=extraction,
    description='ASTER_Features',
    folder='EarthEngine_Dane',
    fileNamePrefix='ASTER_data_features',
    fileFormat='CSV'
)

In [113]:
task.start()

In [115]:
ASTER_df = pd.read_csv('../datasets/ASTER_data_features.csv')
print(ASTER_df.shape)
ASTER_df.head()

(50234, 16)


,system:index,B04,B05,B06,B07,B08,B09,B10,B11,B12,B13,B14,ID,Latitude,Longitude,.geo
0,00000000000000001b6a_0,57.0,38.0,39.5,35.5,32.5,30.5,1252.0,1372.5,1474.5,1730.5,1810.5,JUZSWK,10.53952,-13.4331,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,00000000000000004478_0,57.0,38.0,39.5,35.5,32.5,30.5,1252.0,1372.5,1474.5,1730.5,1810.5,B5RDUR,10.53952,-13.4331,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,0000000000000000444f_0,74.0,50.0,51.0,44.0,41.0,34.0,1391.0,1498.0,1581.0,1859.0,1938.0,IMQ2UL,10.51887,-13.4330,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,00000000000000004450_0,74.0,50.0,51.0,44.0,41.0,34.0,1391.0,1498.0,1581.0,1859.0,1938.0,GAP8LB,10.51887,-13.4330,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,0000000000000000b492_0,74.0,50.0,51.0,44.0,41.0,34.0,1391.0,1498.0,1581.0,1859.0,1938.0,Q7QJ6M,10.51887,-13.4330,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


In [116]:
ASTER_df.drop(columns=['system:index', 'Longitude', 'Latitude', '.geo'], inplace=True)
ASTER_df.head()

,B04,B05,B06,B07,B08,B09,B10,B11,B12,B13,B14,ID
0,57.0,38.0,39.5,35.5,32.5,30.5,1252.0,1372.5,1474.5,1730.5,1810.5,JUZSWK
1,57.0,38.0,39.5,35.5,32.5,30.5,1252.0,1372.5,1474.5,1730.5,1810.5,B5RDUR
2,74.0,50.0,51.0,44.0,41.0,34.0,1391.0,1498.0,1581.0,1859.0,1938.0,IMQ2UL
3,74.0,50.0,51.0,44.0,41.0,34.0,1391.0,1498.0,1581.0,1859.0,1938.0,GAP8LB
4,74.0,50.0,51.0,44.0,41.0,34.0,1391.0,1498.0,1581.0,1859.0,1938.0,Q7QJ6M


In [121]:
olek_features = pd.read_csv('../features/olek-features.csv')
print(olek_features.shape)
olek_features.head()

(50368, 39)


,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,...,phh2o_30-60cm_mean,phh2o_5-15cm_mean,sand_0-5cm_mean,sand_15-30cm_mean,sand_30-60cm_mean,sand_5-15cm_mean,soc_0-5cm_mean,soc_15-30cm_mean,soc_30-60cm_mean,soc_5-15cm_mean
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,1287.0,...,52.0,52.0,508.0,463.0,441.0,502.0,286.0,215.0,181.0,226.0
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3215.0,...,52.0,52.0,527.0,477.0,461.0,514.0,260.0,164.0,137.0,229.0
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3127.0,...,52.0,52.0,527.0,477.0,461.0,514.0,260.0,164.0,137.0,229.0
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,2085.0,...,52.0,52.0,514.0,490.0,448.0,508.0,270.0,158.0,124.0,225.0
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.35551,2059.0,...,52.0,52.0,544.0,501.0,491.0,528.0,233.0,125.0,112.0,190.0


In [123]:
features_comb = pd.merge(olek_features, ASTER_df, on='ID', how='left')
print(features_comb.shape)
features_comb.head()

(50368, 50)


,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11_x,...,B05,B06,B07,B08,B09,B10,B11_y,B12_y,B13,B14
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,1287.0,...,52.5,46.0,49.5,45.0,40.5,1121.0,1214.0,1309.5,1560.5,1637.5
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3215.0,...,58.0,58.5,57.0,54.0,48.5,1179.0,1272.0,1388.0,1635.5,1713.5
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3127.0,...,55.5,57.5,54.0,51.0,44.0,1175.5,1275.0,1389.5,1635.5,1709.0
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,2085.0,...,44.5,46.0,41.5,38.5,31.5,1138.0,1230.5,1328.5,1573.5,1651.0
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.35551,2059.0,...,44.0,47.0,42.0,38.0,38.0,1191.0,1276.0,1363.0,1612.0,1691.0


In [125]:
features_comb.info()

<class 'pandas.DataFrame'>
RangeIndex: 50368 entries, 0 to 50367
Data columns (total 50 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   ID                                 50368 non-null  str    
 1   Country                            50368 non-null  str    
 2   Area                               50368 non-null  str    
 3   Cropland nitrogen per unit area    50368 non-null  float64
 4   Cropland phosphorus per unit area  50368 non-null  float64
 5   Cropland potassium per unit area   50368 non-null  float64
 6   tmin_avg                           50368 non-null  float64
 7   tmax_avg                           50368 non-null  float64
 8   prec_avg                           50368 non-null  float64
 9   B11_x                              50368 non-null  float64
 10  B12_x                              50368 non-null  float64
 11  B2                                 50368 non-null  float64
 12  B

In [126]:
features_comb.fillna(features_comb.mean(numeric_only=True), inplace=True)
features_comb.info()

<class 'pandas.DataFrame'>
RangeIndex: 50368 entries, 0 to 50367
Data columns (total 50 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   ID                                 50368 non-null  str    
 1   Country                            50368 non-null  str    
 2   Area                               50368 non-null  str    
 3   Cropland nitrogen per unit area    50368 non-null  float64
 4   Cropland phosphorus per unit area  50368 non-null  float64
 5   Cropland potassium per unit area   50368 non-null  float64
 6   tmin_avg                           50368 non-null  float64
 7   tmax_avg                           50368 non-null  float64
 8   prec_avg                           50368 non-null  float64
 9   B11_x                              50368 non-null  float64
 10  B12_x                              50368 non-null  float64
 11  B2                                 50368 non-null  float64
 12  B

In [127]:
features_comb.to_csv('../features/olek-features.csv', index=False)

## iSDAsoil features

In [201]:
ee.Initialize(project='lucknera')

In [202]:
ph = ee.Image("ISDASOIL/Africa/v1/ph").select(['mean_0_20'], ['ph'])
clay = ee.Image("ISDASOIL/Africa/v1/clay_content").select(['mean_0_20'], ['clay'])
carbon = ee.Image("ISDASOIL/Africa/v1/carbon_total").select(['mean_0_20'], ['carbon'])

isda_base = ee.Image.cat([ph, clay, carbon])

In [203]:
element_layers = {
    "Al": "aluminium_extractable",
    #Not available "B":  "boron_extractable",
    "Ca": "calcium_extractable",
    #Not available "Cu": "copper_extractable",
    "Fe": "iron_extractable",
    "K":  "potassium_extractable",
    "Mg": "magnesium_extractable",
    #Not available "Mn": "manganese_extractable",
    "N":  "nitrogen_total",      
    #Not available "Na": "sodium_extractable",
    "P":  "phosphorus_extractable",
    "S":  "sulphur_extractable",
    "Zn": "zinc_extractable"
}

In [204]:
images_list = []
for symbol, layer_name in element_layers.items():
    img = ee.Image(f"ISDASOIL/Africa/v1/{layer_name}")
    img_renamed = img.select(['mean_0_20'], [symbol])
    images_list.append(img_renamed)
isda_all = ee.Image.cat([images_list, isda_base])

In [205]:
def create_geometry(feature):
    lon = ee.Number(feature.get('Longitude'))
    lat = ee.Number(feature.get('Latitude'))
    return feature.setGeometry(ee.Geometry.Point([lon, lat]))

In [206]:
points = ee.FeatureCollection("projects/lucknera/assets/olek-sentinel-features")
points_geo = points.map(create_geometry)

In [207]:
extraction_isda = isda_all.sampleRegions(
    collection=points_geo,
    scale=30,
    geometries=True
)

In [208]:
task_isda = ee.batch.Export.table.toDrive(
    collection=extraction_isda,
    description='iSDAsoil_features',
    folder='EarthEngine_Dane', 
    fileFormat='CSV'
)

In [209]:
task_isda.start()

In [212]:
ph = ee.Image("ISDASOIL/Africa/v1/ph").select(['mean_20_50'], ['ph_20_50'])
clay = ee.Image("ISDASOIL/Africa/v1/clay_content").select(['mean_20_50'], ['clay_20_50'])
carbon = ee.Image("ISDASOIL/Africa/v1/carbon_total").select(['mean_20_50'], ['carbon_20_50'])

isda_base = ee.Image.cat([ph, clay, carbon])
element_layers = {
    "Al": "aluminium_extractable",
    #Not available "B":  "boron_extractable",
    "Ca": "calcium_extractable",
    #Not available "Cu": "copper_extractable",
    "Fe": "iron_extractable",
    "K":  "potassium_extractable",
    "Mg": "magnesium_extractable",
    #Not available "Mn": "manganese_extractable",
    "N":  "nitrogen_total",      
    #Not available "Na": "sodium_extractable",
    "P":  "phosphorus_extractable",
    "S":  "sulphur_extractable",
    "Zn": "zinc_extractable"
}
images_list = []
for symbol, layer_name in element_layers.items():
    img = ee.Image(f"ISDASOIL/Africa/v1/{layer_name}")
    img_renamed = img.select(['mean_20_50'], [symbol + "_20_50"])
    images_list.append(img_renamed)
isda_all = ee.Image.cat([images_list, isda_base])
def create_geometry(feature):
    lon = ee.Number(feature.get('Longitude'))
    lat = ee.Number(feature.get('Latitude'))
    return feature.setGeometry(ee.Geometry.Point([lon, lat]))
points = ee.FeatureCollection("projects/lucknera/assets/olek-sentinel-features")
points_geo = points.map(create_geometry)
extraction_isda = isda_all.sampleRegions(
    collection=points_geo,
    scale=30,
    geometries=True
)
task_isda = ee.batch.Export.table.toDrive(
    collection=extraction_isda,
    description='iSDAsoil_features_20_50',
    folder='EarthEngine_Dane', 
    fileFormat='CSV'
)
task_isda.start()

In [225]:
isda_df = pd.read_csv('../datasets/iSDAsoil_features.csv')
print(isda_df.shape)
isda_df.head()

(50365, 17)


,system:index,Al,Ca,Fe,ID,K,Latitude,Longitude,Mg,N,P,S,Zn,carbon,clay,ph,.geo
0,00000000000000001b6a_0,66,67,40,JUZSWK,43,10.53952,-13.4331,49,148,21,38,9,41,26,53,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,00000000000000004478_0,66,67,40,B5RDUR,43,10.53952,-13.4331,49,148,21,38,9,41,26,53,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,0000000000000000444f_0,65,63,40,IMQ2UL,43,10.51887,-13.4330,48,131,22,27,9,39,28,54,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,00000000000000004450_0,65,63,40,GAP8LB,43,10.51887,-13.4330,48,131,22,27,9,39,28,54,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,0000000000000000b492_0,65,63,40,Q7QJ6M,43,10.51887,-13.4330,48,131,22,27,9,39,28,54,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


In [226]:
isda_df_20_50 = pd.read_csv('../datasets/iSDAsoil_features_20_50.csv')
print(isda_df_20_50.shape)
isda_df_20_50.head()

(50365, 17)


,system:index,Al_20_50,Ca_20_50,Fe_20_50,ID,K_20_50,Latitude,Longitude,Mg_20_50,N_20_50,P_20_50,S_20_50,Zn_20_50,carbon_20_50,clay_20_50,ph_20_50,.geo
0,00000000000000001b6a_0,61,64,33,JUZSWK,40,10.53952,-13.4331,48,108,18,33,8,39,29,52,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,00000000000000004478_0,61,64,33,B5RDUR,40,10.53952,-13.4331,48,108,18,33,8,39,29,52,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,0000000000000000444f_0,60,61,32,IMQ2UL,40,10.51887,-13.4330,47,95,19,23,8,36,31,53,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,00000000000000004450_0,60,61,32,GAP8LB,40,10.51887,-13.4330,47,95,19,23,8,36,31,53,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,0000000000000000b492_0,60,61,32,Q7QJ6M,40,10.51887,-13.4330,47,95,19,23,8,36,31,53,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


In [227]:
isda_df.drop(columns=['system:index', 'Longitude', 'Latitude', '.geo'], inplace=True)
isda_df_20_50.drop(columns=['system:index', 'Longitude', 'Latitude', '.geo'], inplace=True)

In [228]:
minerals = ['Al', 'Ca', 'Fe', 'K', 'Mg', 'P', 'S', 'Zn']
minerals_20_50 = [col + "_20_50" for col in minerals]
for col in minerals:
    isda_df[col] = np.exp(isda_df[col] / 10) - 1
for col in minerals_20_50:
    isda_df_20_50[col] = np.exp(isda_df_20_50[col] / 10) - 1

In [229]:
isda_df["N"] = (np.exp(isda_df["N"] / 10) - 1) * 1000
isda_df_20_50["N_20_50"] = (np.exp(isda_df_20_50["N_20_50"] / 10) - 1) * 1000


isda_df['ph'] = isda_df['ph'] / 10
isda_df_20_50['ph_20_50'] = isda_df_20_50['ph_20_50'] / 10

In [231]:
isda_df.head()

,Al,Ca,Fe,ID,K,Mg,N,P,S,Zn,carbon,clay,ph
0,734.095189,811.405825,53.59815,JUZSWK,72.699794,133.289780,2.676444e+09,7.166170,43.701184,1.459603,41,26,5.3
1,734.095189,811.405825,53.59815,B5RDUR,72.699794,133.289780,2.676444e+09,7.166170,43.701184,1.459603,41,26,5.3
2,664.141633,543.571910,53.59815,IMQ2UL,72.699794,120.510418,4.889414e+08,8.025013,13.879732,1.459603,39,28,5.4
3,664.141633,543.571910,53.59815,GAP8LB,72.699794,120.510418,4.889414e+08,8.025013,13.879732,1.459603,39,28,5.4
4,664.141633,543.571910,53.59815,Q7QJ6M,72.699794,120.510418,4.889414e+08,8.025013,13.879732,1.459603,39,28,5.4


In [236]:
isda_df_20_50.columns = isda_df.columns
isda_df_20_50.head()

,Al,Ca,Fe,ID,K,Mg,N,P,S,Zn,carbon,clay,ph
0,444.857770,600.845038,26.112639,JUZSWK,53.59815,120.510418,4.901980e+07,5.049647,26.112639,1.225541,39,29,5.2
1,444.857770,600.845038,26.112639,B5RDUR,53.59815,120.510418,4.901980e+07,5.049647,26.112639,1.225541,39,29,5.2
2,402.428793,444.857770,23.532530,IMQ2UL,53.59815,108.947172,1.335873e+07,5.685894,8.974182,1.225541,36,31,5.3
3,402.428793,444.857770,23.532530,GAP8LB,53.59815,108.947172,1.335873e+07,5.685894,8.974182,1.225541,36,31,5.3
4,402.428793,444.857770,23.532530,Q7QJ6M,53.59815,108.947172,1.335873e+07,5.685894,8.974182,1.225541,36,31,5.3


In [237]:
depth_df = pd.concat([data_train, data_test], ignore_index=True)
depth_df = depth_df[["ID", "Depth_cm"]]
depth_df.head()

,ID,Depth_cm
0,O2TONM,20-50
1,BQLUK6,20-50
2,LSET8M,20-50
3,LEEL7I,20-50
4,LDNGO2,20-50


In [240]:
isda_df_depth = isda_df.merge(depth_df, on='ID', how='left')
isda_df_depth = isda_df_depth.where(isda_df_depth['Depth_cm'] == '0-20')
isda_df_depth.drop(columns=['Depth_cm'], inplace=True)
isda_df_depth.dropna(inplace=True)
print(isda_df_depth.shape)
isda_df_depth.head()

(27871, 13)


,Al,Ca,Fe,ID,K,Mg,N,P,S,Zn,carbon,clay,ph
1,734.095189,811.405825,53.598150,B5RDUR,72.699794,133.289780,2.676444e+09,7.166170,43.701184,1.459603,41.0,26.0,5.3
2,664.141633,543.571910,53.598150,IMQ2UL,72.699794,120.510418,4.889414e+08,8.025013,13.879732,1.459603,39.0,28.0,5.4
3,664.141633,543.571910,53.598150,GAP8LB,72.699794,120.510418,4.889414e+08,8.025013,13.879732,1.459603,39.0,28.0,5.4
6,664.141633,811.405825,53.598150,29L2PF,72.699794,133.289780,1.982758e+09,7.166170,32.115452,1.718282,41.0,25.0,5.3
8,664.141633,402.428793,65.686331,9KOEHL,89.017131,89.017131,1.090968e+08,8.025013,7.166170,2.004166,37.0,24.0,5.1


In [241]:
isda_df_20_50_depth = isda_df_20_50.merge(depth_df, on='ID', how='left')
isda_df_20_50_depth = isda_df_20_50_depth.where(isda_df_20_50_depth['Depth_cm'] == '20-50')
isda_df_20_50_depth.drop(columns=['Depth_cm'], inplace=True)
isda_df_20_50_depth.dropna(inplace=True)
print(isda_df_20_50_depth.shape)
isda_df_20_50_depth.head()

(22494, 13)


,Al,Ca,Fe,ID,K,Mg,N,P,S,Zn,carbon,clay,ph
0,444.857770,600.845038,26.112639,JUZSWK,53.598150,120.510418,4.901980e+07,5.049647,26.112639,1.225541,39.0,29.0,5.2
4,402.428793,444.857770,23.532530,Q7QJ6M,53.598150,108.947172,1.335873e+07,5.685894,8.974182,1.225541,36.0,31.0,5.3
5,402.428793,600.845038,26.112639,3EYUL8,53.598150,108.947172,4.013384e+07,4.473947,21.197951,1.718282,39.0,29.0,5.3
7,444.857770,329.299560,26.112639,UUIOIG,72.699794,80.450869,6.001912e+06,5.685894,4.473947,1.718282,34.0,27.0,5.0
9,491.749041,991.274716,23.532530,S6DD78,53.598150,108.947172,2.196950e+08,5.049647,23.532530,1.459603,41.0,26.0,5.1


In [242]:
isda_combined = pd.concat([isda_df_depth, isda_df_20_50_depth], ignore_index=True)
print(isda_combined.shape)
isda_combined.head()

(50365, 13)


,Al,Ca,Fe,ID,K,Mg,N,P,S,Zn,carbon,clay,ph
0,734.095189,811.405825,53.598150,B5RDUR,72.699794,133.289780,2.676444e+09,7.166170,43.701184,1.459603,41.0,26.0,5.3
1,664.141633,543.571910,53.598150,IMQ2UL,72.699794,120.510418,4.889414e+08,8.025013,13.879732,1.459603,39.0,28.0,5.4
2,664.141633,543.571910,53.598150,GAP8LB,72.699794,120.510418,4.889414e+08,8.025013,13.879732,1.459603,39.0,28.0,5.4
3,664.141633,811.405825,53.598150,29L2PF,72.699794,133.289780,1.982758e+09,7.166170,32.115452,1.718282,41.0,25.0,5.3
4,664.141633,402.428793,65.686331,9KOEHL,89.017131,89.017131,1.090968e+08,8.025013,7.166170,2.004166,37.0,24.0,5.1


In [243]:
isda_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 50365 entries, 0 to 50364
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Al      50365 non-null  float64
 1   Ca      50365 non-null  float64
 2   Fe      50365 non-null  float64
 3   ID      50365 non-null  str    
 4   K       50365 non-null  float64
 5   Mg      50365 non-null  float64
 6   N       50365 non-null  float64
 7   P       50365 non-null  float64
 8   S       50365 non-null  float64
 9   Zn      50365 non-null  float64
 10  carbon  50365 non-null  float64
 11  clay    50365 non-null  float64
 12  ph      50365 non-null  float64
dtypes: float64(12), str(1)
memory usage: 5.0 MB


In [244]:
olek_features = pd.read_csv('../features/olek-features.csv')
print(olek_features.shape)
olek_features.head()

(50368, 50)


,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11_x,...,B05,B06,B07,B08,B09,B10,B11_y,B12_y,B13,B14
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,1287.0,...,52.5,46.0,49.5,45.0,40.5,1121.0,1214.0,1309.5,1560.5,1637.5
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3215.0,...,58.0,58.5,57.0,54.0,48.5,1179.0,1272.0,1388.0,1635.5,1713.5
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,3127.0,...,55.5,57.5,54.0,51.0,44.0,1175.5,1275.0,1389.5,1635.5,1709.0
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878,2085.0,...,44.5,46.0,41.5,38.5,31.5,1138.0,1230.5,1328.5,1573.5,1651.0
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.35551,2059.0,...,44.0,47.0,42.0,38.0,38.0,1191.0,1276.0,1363.0,1612.0,1691.0


In [245]:
olek_features.drop(columns=['Cropland nitrogen per unit area', 'Cropland phosphorus per unit area', 'Cropland potassium per unit area'], inplace=True)
olek_features.head()

,ID,Country,Area,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,...,B05,B06,B07,B08,B09,B10,B11_y,B12_y,B13,B14
0,O2TONM,Tanzania,Tanzania,12.727273,22.628788,123.26878,1287.0,699.0,243.0,364.0,...,52.5,46.0,49.5,45.0,40.5,1121.0,1214.0,1309.5,1560.5,1637.5
1,BQLUK6,Tanzania,Tanzania,12.727273,22.628788,123.26878,3215.0,2426.0,789.0,1066.0,...,58.0,58.5,57.0,54.0,48.5,1179.0,1272.0,1388.0,1635.5,1713.5
2,LSET8M,Tanzania,Tanzania,12.727273,22.628788,123.26878,3127.0,2220.0,697.0,943.0,...,55.5,57.5,54.0,51.0,44.0,1175.5,1275.0,1389.5,1635.5,1709.0
3,LEEL7I,Tanzania,Tanzania,12.727273,22.628788,123.26878,2085.0,1370.0,499.0,691.0,...,44.5,46.0,41.5,38.5,31.5,1138.0,1230.5,1328.5,1573.5,1651.0
4,LDNGO2,Tanzania,Tanzania,12.674242,23.077652,110.35551,2059.0,1076.0,324.0,436.0,...,44.0,47.0,42.0,38.0,38.0,1191.0,1276.0,1363.0,1612.0,1691.0


In [257]:
features_comb = pd.merge(olek_features, isda_combined, on='ID', how='left')
print(features_comb.shape)
features_comb.head()

(50368, 59)


,ID,Country,Area,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,...,Fe,K,Mg,N,P,S,Zn,carbon,clay,ph
0,O2TONM,Tanzania,Tanzania,12.727273,22.628788,123.26878,1287.0,699.0,243.0,364.0,...,28.964100,133.289780,120.510418,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6
1,BQLUK6,Tanzania,Tanzania,12.727273,22.628788,123.26878,3215.0,2426.0,789.0,1066.0,...,26.112639,133.289780,108.947172,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7
2,LSET8M,Tanzania,Tanzania,12.727273,22.628788,123.26878,3127.0,2220.0,697.0,943.0,...,28.964100,133.289780,120.510418,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7
3,LEEL7I,Tanzania,Tanzania,12.727273,22.628788,123.26878,2085.0,1370.0,499.0,691.0,...,26.112639,108.947172,133.289780,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5
4,LDNGO2,Tanzania,Tanzania,12.674242,23.077652,110.35551,2059.0,1076.0,324.0,436.0,...,23.532530,147.413159,98.484316,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0


In [258]:
features_comb.fillna(features_comb.mean(numeric_only=True), inplace=True)
features_comb.info()

<class 'pandas.DataFrame'>
RangeIndex: 50368 entries, 0 to 50367
Data columns (total 59 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  50368 non-null  str    
 1   Country             50368 non-null  str    
 2   Area                50368 non-null  str    
 3   tmin_avg            50368 non-null  float64
 4   tmax_avg            50368 non-null  float64
 5   prec_avg            50368 non-null  float64
 6   B11_x               50368 non-null  float64
 7   B12_x               50368 non-null  float64
 8   B2                  50368 non-null  float64
 9   B3                  50368 non-null  float64
 10  B4                  50368 non-null  float64
 11  B8                  50368 non-null  float64
 12  aspect              50368 non-null  int64  
 13  elevation           50368 non-null  int64  
 14  hillshade           50368 non-null  int64  
 15  slope               50368 non-null  int64  
 16  cec_0-5cm_mean 

In [259]:
features_comb.columns

Index(['ID', 'Country', 'Area', 'tmin_avg', 'tmax_avg', 'prec_avg', 'B11_x',
       'B12_x', 'B2', 'B3', 'B4', 'B8', 'aspect', 'elevation', 'hillshade',
       'slope', 'cec_0-5cm_mean', 'cec_15-30cm_mean', 'cec_30-60cm_mean',
       'cec_5-15cm_mean', 'clay_0-5cm_mean', 'clay_15-30cm_mean',
       'clay_30-60cm_mean', 'clay_5-15cm_mean', 'phh2o_0-5cm_mean',
       'phh2o_15-30cm_mean', 'phh2o_30-60cm_mean', 'phh2o_5-15cm_mean',
       'sand_0-5cm_mean', 'sand_15-30cm_mean', 'sand_30-60cm_mean',
       'sand_5-15cm_mean', 'soc_0-5cm_mean', 'soc_15-30cm_mean',
       'soc_30-60cm_mean', 'soc_5-15cm_mean', 'B04', 'B05', 'B06', 'B07',
       'B08', 'B09', 'B10', 'B11_y', 'B12_y', 'B13', 'B14', 'Al', 'Ca', 'Fe',
       'K', 'Mg', 'N', 'P', 'S', 'Zn', 'carbon', 'clay', 'ph'],
      dtype='str')

In [268]:
minerals = ['Al', 'Ca', 'Fe', 'K', 'Mg', 'P', 'S', 'Zn', 'N']

In [269]:
names = [f"{c}_pred" for c in minerals]
features_comb.rename(columns=dict(zip(minerals, names)), inplace=True)
features_comb.head()

,ID,Country,Area,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,...,Fe_pred,K_pred,Mg_pred,N_pred,P_pred,S_pred,Zn_pred,carbon,clay,ph
0,O2TONM,Tanzania,Tanzania,12.727273,22.628788,123.26878,1287.0,699.0,243.0,364.0,...,28.964100,133.289780,120.510418,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6
1,BQLUK6,Tanzania,Tanzania,12.727273,22.628788,123.26878,3215.0,2426.0,789.0,1066.0,...,26.112639,133.289780,108.947172,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7
2,LSET8M,Tanzania,Tanzania,12.727273,22.628788,123.26878,3127.0,2220.0,697.0,943.0,...,28.964100,133.289780,120.510418,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7
3,LEEL7I,Tanzania,Tanzania,12.727273,22.628788,123.26878,2085.0,1370.0,499.0,691.0,...,26.112639,108.947172,133.289780,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5
4,LDNGO2,Tanzania,Tanzania,12.674242,23.077652,110.35551,2059.0,1076.0,324.0,436.0,...,23.532530,147.413159,98.484316,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0


In [270]:
features_comb.to_csv('../features/olek-features.csv', index=False)

## Lithology

In [271]:
import ee

In [279]:
ee.Initialize(project='lucknera')

In [280]:
landforms_img = ee.Image("CSP/ERGo/1_0/Global/ALOS_landforms")
landforms_renamed = landforms_img.select(['constant'], ['landform_class'])

In [281]:
extraction_landforms = landforms_renamed.sampleRegions(
    collection=points_geo,
    scale=90,
    geometries=True
)

In [282]:
task_landforms = ee.batch.Export.table.toDrive(
    collection=extraction_landforms,
    description='landforms_features',
    folder='EarthEngine_Dane', 
    fileFormat='CSV'
)

In [283]:
task_landforms.start()

In [304]:
landforms_df = pd.read_csv('../features/landforms_features.csv')
print(landforms_df.shape)
landforms_df.head()

(50368, 6)


,system:index,ID,Latitude,Longitude,landform_class,.geo
0,00000000000000001b6a_0,JUZSWK,10.53952,-13.4331,21,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,00000000000000004478_0,B5RDUR,10.53952,-13.4331,21,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,0000000000000000444f_0,IMQ2UL,10.51887,-13.4330,21,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,00000000000000004450_0,GAP8LB,10.51887,-13.4330,21,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,0000000000000000b492_0,Q7QJ6M,10.51887,-13.4330,21,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


In [305]:
landforms_df.drop(columns=['system:index', 'Latitude', 'Longitude', '.geo'], inplace=True, errors='ingore')

In [306]:
landforms_df.head()

,ID,landform_class
0,JUZSWK,21
1,B5RDUR,21
2,IMQ2UL,21
3,GAP8LB,21
4,Q7QJ6M,21


In [307]:
landforms_df['landform_class'].unique()

array([21, 31, 41, 11, 42, 34, 24, 12, 22, 14])

In [308]:
landforms_df = pd.get_dummies(landforms_df, columns=['landform_class'], prefix='landform', dtype=int)

In [309]:
landforms_df.head()

,ID,landform_11,landform_12,landform_14,landform_21,landform_22,landform_24,landform_31,landform_34,landform_41,landform_42
0,JUZSWK,0,0,0,1,0,0,0,0,0,0
1,B5RDUR,0,0,0,1,0,0,0,0,0,0
2,IMQ2UL,0,0,0,1,0,0,0,0,0,0
3,GAP8LB,0,0,0,1,0,0,0,0,0,0
4,Q7QJ6M,0,0,0,1,0,0,0,0,0,0


In [310]:
landforms_df.to_csv('../features/landforms_features.csv')

## Sentinel features v2

In [311]:
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(points_geo)
                 .filterDate('2021-01-01', '2021-12-31') 
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

In [312]:
s2_median = s2_collection.median().select(['B5', 'B7'])

In [313]:
cire_index = s2_median.expression(
    '(B7 / B5) - 1', {
        'B7': s2_median.select('B7'),
        'B5': s2_median.select('B5')
    }
).rename('CIre')

In [314]:
s2_final = s2_median.addBands(cire_index)

In [315]:
extraction_s2 = s2_final.sampleRegions(
    collection=points_geo,
    scale=20, 
    geometries=True
)

In [316]:
task_s2 = ee.batch.Export.table.toDrive(
    collection=extraction_s2,
    description='CIre-feature',
    folder='EarthEngine_Dane', 
    fileFormat='CSV'
)

In [317]:
task_s2.start()

In [320]:
cire_df = pd.read_csv('../features/CIre-feature.csv')
cire_df.head()

,system:index,B5,B7,CIre,ID,Latitude,Longitude,.geo
0,00000000000000001b6a_0,1101.0,2610.0,1.370572,JUZSWK,10.53952,-13.4331,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,00000000000000004478_0,1101.0,2610.0,1.370572,B5RDUR,10.53952,-13.4331,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,0000000000000000444f_0,1037.0,2373.0,1.288332,IMQ2UL,10.51887,-13.4330,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,00000000000000004450_0,1037.0,2373.0,1.288332,GAP8LB,10.51887,-13.4330,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,0000000000000000b492_0,1037.0,2373.0,1.288332,Q7QJ6M,10.51887,-13.4330,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


In [321]:
cire_df.drop(columns=['system:index', 'B5', 'B7', 'Latitude', 'Longitude', '.geo'], inplace=True)
cire_df.head()

,CIre,ID
0,1.370572,JUZSWK
1,1.370572,B5RDUR
2,1.288332,IMQ2UL
3,1.288332,GAP8LB
4,1.288332,Q7QJ6M


In [322]:
cire_df.to_csv('../features/CIre-feature.csv')